In [21]:
!uv pip install peft

Using Python 3.11.11 environment at: /Users/bennetjollenbeck/.pyenv/versions/3.11.11/envs/grocery-list-venv
Audited 1 package in 6ms


In [ ]:
# server/classifier/classifier.py
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

INSTRUCTION = "instruction": "extract this into one of these categories('meat_fish', 'fruit_veg', 'bread_bakery', 'grains_pasta', 'pantry_staples', 'spices_seasoning', 'snacks', 'drinks', 'frozen', 'household', 'canned_jars', 'dairy_eggs')"
class GroceryClassifier:
    def __init__(self):
        base_model = "NousResearch/Llama-3.2-1B"
        adapter_path = "./outputs/lora-out"
        tokenizer = AutoTokenizer.from_pretrained(adapter_path)
        model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float32)
        self.model = PeftModel.from_pretrained(model, adapter_path).eval()
        self.tokenizer = tokenizer

    def classify(self, item: str) -> str:
        prompt = f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{item}\n\n### Response:\n"
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=20, do_sample=False)
        result = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return result.split("### Response:")[-1].strip()

# Load once at startup
classifier = GroceryClassifier()

ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?

In [ ]:
# construct dataset 

import json
with open("dataset.json", "r") as file:
    data = json.load(file)

result = {
    "input": [],
    "answer": []
}

for entry in data:
    
    result["input"].append(entry["input"])
    result["answer"].append(entry["answer"])

instruct = """extract this into one of these categories( 
        'meat_fish',       
        'fruit_veg',        
        'bread_bakery',     
        'grains_pasta',     
        'pantry_staples',   
        'spices_seasoning', 
        'snacks',           
        'drinks',           
        'frozen',           
        'household',        
        'canned_jars'
        )"""
jsonLines = [ str({
    "instruction": instruct,
    "input": result["input"][i],
    "answer": result["answer"][i]
}) for i in range(len(list(result.values())))]

with open("dataset.json", "w") as file:
    file.writelines(jsonLines)
df = pd.DataFrame(result)
df.head()

JSONDecodeError: Extra data: line 5 column 4 (char 288)

In [ ]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

Loading weights: 100%|██████████| 146/146 [00:17<00:00,  8.49it/s, Materializing param=model.norm.weight]                              
